# Fase 5 · M01: Modelos Lineales — Comparativa Completa

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Notebook** | f5_m01_lineales.ipynb |

## Qué hace
Carga los resultados de `f5_m01_lineales_basico.ipynb` (4 modelos) y
`f5_m01_lineales_ext.ipynb` (3 modelos), los combina y genera:
- `m01_lineales.html` — comparativa completa 7 modelos (botón principal)
- `m01_discriminativos.html` — LogReg · SVM_lineal · SVM_RBF · Perceptron · SGD
- `m01_generativos.html` — LDA · Ridge
- `results_lineales_completo.parquet` + `.xlsx`

## Requisitos
Ejecutar antes: `f5_m01_lineales_basico.ipynb` y `f5_m01_lineales_ext.ipynb`

## Genera
- `docs/html/fase5/m01_lineales.html`
- `docs/html/fase5/m01_discriminativos.html`
- `docs/html/fase5/m01_generativos.html`
- `data/05_modelado/results/results_lineales_completo.parquet`
- `data/05_modelado/results/results_lineales_completo.xlsx`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys, json, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
import io, base64

warnings.filterwarnings('ignore')

# ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import formato_numero_es as fmt, crear_directorios
from src.html.render import render_pagina

RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_HTML_F5])

COLOR = '#3182ce'

def figura_a_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')


✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR Y COMBINAR RESULTADOS
# ============================================================================
# Lee los parquets de basico y ext, añade columna 'grupo' y los une.
# ============================================================================

ruta_basico = RUTA_RESULTS / 'results_lineales_basico.parquet'
ruta_ext    = RUTA_RESULTS / 'results_lineales_ext.parquet'

if not ruta_basico.exists():
    raise FileNotFoundError(f'Ejecuta primero f5_m01_lineales_basico.ipynb\n{ruta_basico}')
if not ruta_ext.exists():
    raise FileNotFoundError(f'Ejecuta primero f5_m01_lineales_ext.ipynb\n{ruta_ext}')

df_basico = pd.read_parquet(ruta_basico)
df_ext    = pd.read_parquet(ruta_ext)

df_basico['grupo'] = 'basico'
df_ext['grupo']    = 'ext'

df_total = pd.concat([df_basico, df_ext], ignore_index=True)
df_total['familia'] = 'lineal'

# Ordenar por AUC descendente
df_total = df_total.sort_values('auc_mean', ascending=False).reset_index(drop=True)

# Mejor por modelo (para tablas pivotadas)
df_mejor_por_modelo = (
    df_total.sort_values('auc_mean', ascending=False)
    .groupby('modelo', sort=False)
    .first()
    .reset_index()
)

MEJOR_GLOBAL     = df_total.iloc[0]
MEJOR_MODELO     = MEJOR_GLOBAL['modelo']
MEJOR_ESTRATEGIA = MEJOR_GLOBAL['estrategia']

# Grupos por tipo
DISCRIMINATIVOS = ['LogReg', 'SVM_lineal', 'SVM_RBF', 'Perceptron', 'SGD']
GENERATIVOS     = ['LDA', 'Ridge']

print(f'📊 Total combinaciones: {len(df_total)} (basico={len(df_basico)}, ext={len(df_ext)})')
print(f'🏆 Mejor global: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC = {MEJOR_GLOBAL["auc_mean"]:.4f} ± {MEJOR_GLOBAL["auc_std"]:.4f}')
print(f'   F1  = {MEJOR_GLOBAL["f1_mean"]:.4f}')


📊 Total combinaciones: 21 (basico=12, ext=9)
🏆 Mejor global: SVM_RBF + balanced
   AUC = 0.9056 ± 0.0034
   F1  = 0.7565


In [3]:
# ============================================================================
# CELDA 3: GUARDAR FICHERO COMPLETO
# ============================================================================
df_total.to_parquet(RUTA_RESULTS / 'results_lineales_completo.parquet', index=False)
df_total.to_excel(RUTA_RESULTS  / 'results_lineales_completo.xlsx',    index=False)
print('✅ results_lineales_completo.parquet guardado')
print('✅ results_lineales_completo.xlsx guardado')
print(f'   {len(df_total)} filas × {len(df_total.columns)} columnas')


✅ results_lineales_completo.parquet guardado
✅ results_lineales_completo.xlsx guardado
   21 filas × 20 columnas


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS COMPARATIVOS (7 modelos)
# ============================================================================
graficos_b64 = {}

# --- Gráfico 1: AUC por modelo (mejor estrategia) ---
modelos_ord = df_mejor_por_modelo.sort_values('auc_mean', ascending=True)
colores     = ['#e53e3e' if m == MEJOR_MODELO else COLOR
               for m in modelos_ord['modelo']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(modelos_ord['modelo'], modelos_ord['auc_mean'], color=colores, height=0.6)
for bar, val in zip(bars, modelos_ord['auc_mean']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('AUC-ROC CV (mejor estrategia)')
ax.set_title('Comparativa AUC — 7 modelos lineales\n(rojo = mejor global)', fontweight='bold')
ax.set_xlim(0.84, 0.93)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['auc_comparativa'] = figura_a_base64(fig)
plt.close(fig)

# --- Gráfico 2: F1 por modelo ---
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(modelos_ord['modelo'],
               df_mejor_por_modelo.set_index('modelo').loc[modelos_ord['modelo'], 'f1_mean'],
               color=colores, height=0.6)
for bar, val in zip(bars, df_mejor_por_modelo.set_index('modelo').loc[modelos_ord['modelo'], 'f1_mean']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('F1 CV (mejor estrategia)')
ax.set_title('Comparativa F1 — 7 modelos lineales\n(rojo = mejor global)', fontweight='bold')
ax.set_xlim(0.68, 0.80)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['f1_comparativa'] = figura_a_base64(fig)
plt.close(fig)

# --- Gráfico 3: AUC vs F1 scatter ---
fig, ax = plt.subplots(figsize=(8, 6))
for _, row in df_mejor_por_modelo.iterrows():
    color = '#e53e3e' if row['modelo'] == MEJOR_MODELO else COLOR
    ax.scatter(row['auc_mean'], row['f1_mean'], s=120, color=color, zorder=5)
    ax.annotate(row['modelo'], (row['auc_mean'], row['f1_mean']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('AUC-ROC CV')
ax.set_ylabel('F1 CV')
ax.set_title('AUC vs F1 — 7 modelos lineales\n(mejor estrategia de cada uno)', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['auc_vs_f1'] = figura_a_base64(fig)
plt.close(fig)

print('✅ 3 gráficos generados')


✅ 3 gráficos generados


In [5]:
# ============================================================================
# CELDA 5: FUNCIÓN PARA GENERAR HTML DE CUALQUIER SUBCONJUNTO
# ============================================================================

def generar_html_subconjunto(df_sub, titulo_seccion, nombre_fichero, nombre_html):
    """Genera HTML para un subconjunto de modelos."""
    df_sub = df_sub.sort_values('auc_mean', ascending=False)
    mejor  = df_sub.iloc[0]
    segundo = df_sub.iloc[1] if len(df_sub) > 1 else df_sub.iloc[0]

    n_modelos = df_sub['modelo'].nunique()
    n_combs   = len(df_sub)

    # KPIs
    kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
        f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
        f'padding:18px 28px;text-align:center;min-width:120px;">'
        f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{v}</div>'
        f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{t}</div></div>'
        for v, t in [
            (str(n_modelos), 'Modelos'),
            ('3', 'Estrategias'),
            (str(n_combs), 'Combinaciones'),
            (f'{mejor["auc_mean"]:.3f}', 'Mejor AUC CV'),
            (f'{mejor["f1_mean"]:.3f}',  'Mejor F1 CV'),
        ]
    ) + '</div>'

    # Texto dinámico
    diff_auc = mejor['auc_mean'] - segundo['auc_mean']
    texto = (
        f'<p>De las <strong>{n_combs} combinaciones</strong> evaluadas ({n_modelos} modelos x 3 estrategias), '
        f'<strong>{mejor["modelo"]}</strong> con estrategia <strong>{mejor["estrategia"]}</strong> '
        f'obtiene el mejor resultado con <strong>AUC CV = {mejor["auc_mean"]:.4f} '
        f'&plusmn; {mejor["auc_std"]:.4f}</strong> y <strong>F1 = {mejor["f1_mean"]:.4f}</strong>.</p>'
        f'<p>Supera al segundo ({segundo["modelo"]} + {segundo["estrategia"]}) '
        f'en <strong>{diff_auc:.4f} puntos de AUC</strong>.</p>'
    )

    # Tabla pivotada
    df_pivot = (
        df_sub.sort_values('auc_mean', ascending=False)
        .groupby('modelo', sort=False).first().reset_index()
    )
    filas_pivot = ''
    for _, r in df_pivot.iterrows():
        bg = 'background:#ebf8ff;font-weight:600;' if r['modelo'] == mejor['modelo'] else ''
        estrella = ' 🏆' if r['modelo'] == mejor['modelo'] else ''
        filas_pivot += (
            f'<tr style="{bg}"><td>{r["modelo"]}{estrella}</td>'
            f'<td>{r["estrategia"]}</td>'
            f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
            f'<td>{r["f1_mean"]:.4f}</td>'
            f'<td>{r["precision_mean"]:.4f}</td>'
            f'<td>{r["recall_mean"]:.4f}</td></tr>'
        )
    tabla_pivot = (
        '<table class="tabla-datos"><thead><tr>'
        '<th>Modelo</th><th>Mejor estrategia</th>'
        '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
        '<th>Precision</th><th>Recall</th></tr></thead>'
        f'<tbody>{filas_pivot}</tbody></table>'
    )

    # Tabla completa
    filas_cv = ''
    for _, r in df_sub.iterrows():
        bg = 'background:#ebf8ff;font-weight:600;' if (
            r['modelo'] == mejor['modelo'] and r['estrategia'] == mejor['estrategia']) else ''
        filas_cv += (
            f'<tr style="{bg}"><td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
            f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
            f'<td>{r["f1_mean"]:.4f}</td>'
            f'<td>{r["precision_mean"]:.4f}</td>'
            f'<td>{r["recall_mean"]:.4f}</td>'
            f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
        )
    tabla_cv = (
        '<table class="tabla-datos"><thead><tr>'
        '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
        '<th>F1</th><th>Precision</th><th>Recall</th><th>Tiempo</th></tr></thead>'
        f'<tbody>{filas_cv}</tbody></table>'
    )

    secciones = (
        f'<section class="seccion"><h2>{titulo_seccion}</h2>{kpis_html}</section>'
        f'<section class="seccion"><h2>Conclusiones</h2>{texto}</section>'
        f'<section class="seccion"><h2>Comparativa por modelo</h2>{tabla_pivot}</section>'
        f'<section class="seccion"><h2>Resultados completos CV</h2>{tabla_cv}</section>'
    )
    render_pagina(nombre_fichero, secciones,
                  RUTA_HTML_F5 / nombre_html,
                  carpeta_notebook='fase5_modelado')
    print(f'✅ {nombre_html} generado')

print('✅ Función generar_html_subconjunto definida')


✅ Función generar_html_subconjunto definida


In [6]:
# ============================================================================
# CELDA 6: GENERAR LOS 3 HTML
# ============================================================================

# Botones de navegación entre los 5 HTMLs de la familia lineal
nav_lineales = '''
<div style="display:flex;flex-wrap:wrap;gap:10px;margin:20px 0;">
  <div style="width:100%;font-weight:600;color:#555;margin-bottom:4px;">
    [A] Por notebook:</div>
  <a href="m01_lineales_basico.html" style="background:#3182ce;color:white;
     padding:8px 16px;border-radius:6px;text-decoration:none;font-size:0.9rem;">
    📊 Basicos (4)</a>
  <a href="m01_lineales_ext.html" style="background:#3182ce;color:white;
     padding:8px 16px;border-radius:6px;text-decoration:none;font-size:0.9rem;">
    ➕ Extendidos (3)</a>
  <div style="width:100%;font-weight:600;color:#555;margin:8px 0 4px;">
    [B] Por tipo de modelo:</div>
  <a href="m01_discriminativos.html" style="background:#553c9a;color:white;
     padding:8px 16px;border-radius:6px;text-decoration:none;font-size:0.9rem;">
    📐 Discriminativos (5)</a>
  <a href="m01_generativos.html" style="background:#553c9a;color:white;
     padding:8px 16px;border-radius:6px;text-decoration:none;font-size:0.9rem;">
    📊 Generativos/otros (2)</a>
</div>
'''

# Gráficos combinados
graficos_html = (
    f'<img src="data:image/png;base64,{graficos_b64["auc_comparativa"]}" style="max-width:100%">'
    f'<img src="data:image/png;base64,{graficos_b64["f1_comparativa"]}" style="max-width:100%">'
    f'<img src="data:image/png;base64,{graficos_b64["auc_vs_f1"]}" style="max-width:100%">'
)

# ── HTML 1: m01_lineales.html (completo 7 modelos + botones + gráficos) ──────
mejor = df_total.iloc[0]
segundo = df_total.iloc[1]
diff_auc = mejor['auc_mean'] - segundo['auc_mean']

df_pivot_total = (
    df_total.sort_values('auc_mean', ascending=False)
    .groupby('modelo', sort=False).first().reset_index()
)
filas_pivot = ''
for _, r in df_pivot_total.iterrows():
    bg = 'background:#ebf8ff;font-weight:600;' if r['modelo'] == MEJOR_MODELO else ''
    estrella = ' 🏆' if r['modelo'] == MEJOR_MODELO else ''
    filas_pivot += (
        f'<tr style="{bg}"><td>{r["modelo"]}{estrella}</td>'
        f'<td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["grupo"]}</td></tr>'
    )
tabla_pivot_total = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
    '<th>Precision</th><th>Recall</th><th>Grupo</th></tr></thead>'
    f'<tbody>{filas_pivot}</tbody></table>'
)

filas_total = ''
for _, r in df_total.iterrows():
    bg = 'background:#ebf8ff;font-weight:600;' if (
        r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA) else ''
    filas_total += (
        f'<tr style="{bg}"><td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td>'
        f'<td>{r["grupo"]}</td></tr>'
    )
tabla_total = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
    '<th>F1</th><th>Precision</th><th>Recall</th><th>Tiempo</th><th>Grupo</th></tr></thead>'
    f'<tbody>{filas_total}</tbody></table>'
)

kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{v}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{t}</div></div>'
    for v, t in [
        ('7', 'Modelos'),
        ('3', 'Estrategias'),
        (str(len(df_total)), 'Combinaciones'),
        (f'{mejor["auc_mean"]:.3f}', 'Mejor AUC CV'),
        (f'{mejor["f1_mean"]:.3f}',  'Mejor F1 CV'),
    ]
) + '</div>'

texto_global = (
    f'<p>La familia lineal completa incluye <strong>7 modelos</strong> evaluados en '
    f'<strong>{len(df_total)} combinaciones</strong> (3 estrategias de balance cada uno). '
    f'El mejor clasificador lineal es <strong>{MEJOR_MODELO}</strong> con estrategia '
    f'<strong>{MEJOR_ESTRATEGIA}</strong>, alcanzando un '
    f'<strong>AUC CV de {mejor["auc_mean"]:.4f} &plusmn; {mejor["auc_std"]:.4f}</strong> '
    f'y <strong>F1 de {mejor["f1_mean"]:.4f}</strong>.</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) '
    f'en <strong>{diff_auc:.4f} puntos de AUC</strong>.</p>'
)

secciones_total = (
    '<section class="seccion"><h2>Resumen familia lineal</h2>' + kpis_html + '</section>'
    '<section class="seccion"><h2>Navegacion</h2>' + nav_lineales + '</section>'
    '<section class="seccion"><h2>Conclusiones</h2>' + texto_global + '</section>'
    '<section class="seccion"><h2>Graficos comparativos — 7 modelos</h2>' + graficos_html + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo (mejor estrategia)</h2>' + tabla_pivot_total + '</section>'
    '<section class="seccion"><h2>Resultados completos — 21 combinaciones</h2>' + tabla_total + '</section>'
)

render_pagina('f5_m01_lineales.ipynb', secciones_total,
              RUTA_HTML_F5 / 'm01_lineales.html',
              carpeta_notebook='fase5_modelado')
print('✅ m01_lineales.html generado (completo 7 modelos)')

# ── HTML 2: discriminativos ───────────────────────────────────────────────────
df_disc = df_total[df_total['modelo'].isin(DISCRIMINATIVOS)]
generar_html_subconjunto(
    df_disc,
    'Modelos discriminativos — LogReg · SVM_lineal · SVM_RBF · Perceptron · SGD',
    'f5_m01_lineales.ipynb',
    'm01_discriminativos.html'
)

# ── HTML 3: generativos ───────────────────────────────────────────────────────
df_gen = df_total[df_total['modelo'].isin(GENERATIVOS)]
generar_html_subconjunto(
    df_gen,
    'Modelos generativos/otros — LDA · Ridge',
    'f5_m01_lineales.ipynb',
    'm01_generativos.html'
)


✅ m01_lineales.html generado (completo 7 modelos)
✅ m01_discriminativos.html generado
✅ m01_generativos.html generado


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================
print('=' * 60)
print('RESUMEN FINAL — f5_m01_lineales (combinado)')
print('=' * 60)
print(f'Modelos     : 7 (4 basicos + 3 ext)')
print(f'Estrategias : none · balanced · smote  (21 combinaciones)')
print()
print(f'🏆 Mejor global : {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_total.iloc[0]["auc_mean"]:.4f} ± {df_total.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_total.iloc[0]["f1_mean"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_lineales_completo.parquet')
print('   data/05_modelado/results/results_lineales_completo.xlsx')
print('   docs/html/fase5/m01_lineales.html          (boton principal)')
print('   docs/html/fase5/m01_lineales_basico.html')
print('   docs/html/fase5/m01_lineales_ext.html')
print('   docs/html/fase5/m01_discriminativos.html')
print('   docs/html/fase5/m01_generativos.html')
print()
print('➡️  Siguiente: f5_m02_arboles.ipynb')


RESUMEN FINAL — f5_m01_lineales (combinado)
Modelos     : 7 (4 basicos + 3 ext)
Estrategias : none · balanced · smote  (21 combinaciones)

🏆 Mejor global : SVM_RBF + balanced
   AUC CV = 0.9056 ± 0.0034
   F1  CV = 0.7565

📁 Archivos:
   data/05_modelado/results/results_lineales_completo.parquet
   data/05_modelado/results/results_lineales_completo.xlsx
   docs/html/fase5/m01_lineales.html          (boton principal)
   docs/html/fase5/m01_lineales_basico.html
   docs/html/fase5/m01_lineales_ext.html
   docs/html/fase5/m01_discriminativos.html
   docs/html/fase5/m01_generativos.html

➡️  Siguiente: f5_m02_arboles.ipynb
